# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import ta
import os

## 1.2 Функции

In [15]:
def filter_trading_hours(df: pd.DataFrame) -> pd.DataFrame:
    """
    Оставляет только данные для времен 10:00, 12:00, 14:00, 16:00, 18:00
    """
    if 'begin' not in df.columns:
        return df
    
    df['begin'] = pd.to_datetime(df['begin'])
    target_hours = [10, 12, 14, 16, 18]
    df_filtered = df[df['begin'].dt.hour.isin(target_hours)].copy()
    
    return df_filtered

def generate_currency_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Генерирует 10 самых важных признаков на основе цены закрытия
    """
    if df.empty or 'close' not in df.columns:
        return df
    
    df_processed = df.copy().sort_values('begin')
    
    # 1. Скользящие средние (тренд)
    df_processed['MA_20'] = df_processed['close'].rolling(window=20).mean()
    df_processed['MA_50'] = df_processed['close'].rolling(window=50).mean()
    
    # 2. RSI (моментум)
    df_processed['RSI_14'] = ta.momentum.RSIIndicator(df_processed['close'], window=14).rsi()
    
    # 3. MACD (тренд + моментум)
    macd = ta.trend.MACD(df_processed['close'])
    df_processed['MACD'] = macd.macd()
    df_processed['MACD_SIGNAL'] = macd.macd_signal()
    
    # 4. Волатильность
    returns = df_processed['close'].pct_change()
    df_processed['VOLATILITY_20'] = returns.rolling(window=20).std() * np.sqrt(252)
    
    # 5. Процентные изменения
    df_processed['CHANGE_1D'] = df_processed['close'].pct_change(1)
    df_processed['CHANGE_5D'] = df_processed['close'].pct_change(5)
    
    # 6. Относительная позиция в диапазоне
    df_processed['RANGE_POSITION'] = (
        (df_processed['close'] - df_processed['close'].rolling(20).min()) / 
        (df_processed['close'].rolling(20).max() - df_processed['close'].rolling(20).min())
    )
    
    # Удаляем строки с пропусками
    df_clean = df_processed.dropna()
    
    return df_clean

def process_currency_data(df: pd.DataFrame, save_path: str) -> pd.DataFrame:
    """
    Обрабатывает DataFrame с валютными данными и сохраняет результат
    
    Args:
        df: DataFrame с колонками 'begin' и 'close'
        save_path: Полный путь для сохранения CSV файла (например, 'path/to/file.csv')
    
    Returns:
        Обработанный DataFrame с признаками
    """
    
    print(f"Исходные данные: {len(df)} строк")
    
    # Фильтруем по времени
    df_filtered = filter_trading_hours(df)
    print(f"После фильтрации времени: {len(df_filtered)} строк")
    
    # Генерируем признаки
    df_features = generate_currency_features(df_filtered)
    print(f"После генерации признаков: {len(df_features)} строк")
    print(f"Количество признаков: {len(df_features.columns)}")
    
    # Сохраняем результат
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df_features.to_csv(save_path, index=False, encoding='utf-8')
    print(f"Данные сохранены в: {save_path}")
    
    return df_features

# 2 Подготовка данных

In [17]:
usd = pd.read_csv("../../data/additional_prices/USDRUBF.csv", parse_dates=['begin'])
usd.head()

,begin,open,close,high,low,value,volume,ticker
0,2022-05-31 10:00:00,65.07,65.12,66.79,63.21,0,3811,USDRUBF
1,2022-05-31 11:00:00,65.11,64.36,65.40,64.30,0,1768,USDRUBF
2,2022-05-31 12:00:00,64.42,64.33,65.10,64.28,0,1343,USDRUBF
3,2022-05-31 13:00:00,64.33,64.11,64.48,64.01,0,731,USDRUBF
4,2022-05-31 14:00:00,64.24,64.67,64.80,64.16,0,567,USDRUBF


In [19]:
cny = pd.read_csv("../../data/additional_prices/CNYRUB_TOM.csv", parse_dates=['begin'])
eur = pd.read_csv("../../data/additional_prices/EURRUBF.csv", parse_dates=['begin'])
gld = pd.read_csv("../../data/additional_prices/GLDRUB_TOM.csv", parse_dates=['begin'])
brent = pd.read_csv("../../data/additional_prices/brent_continuous.csv", parse_dates=['begin'])

## 2.2 Генерация признаков и сохранение 

In [21]:
processed_df = process_currency_data(
    df=usd,
    save_path='../../data/additional_features/USDRUBF_features.csv'
)

Исходные данные: 8704 строк
После фильтрации времени: 4398 строк
После генерации признаков: 4349 строк
Количество признаков: 17
Данные сохранены в: ../../data/additional_features/USDRUBF_features.csv


In [23]:
processed_df = process_currency_data(
    df=cny,
    save_path='../../data/additional_features/CNYRUB_TOM_features.csv'
)
processed_df = process_currency_data(
    df=eur,
    save_path='../../data/additional_features/EURRUBF_features.csv'
)
processed_df = process_currency_data(
    df=gld,
    save_path='../../data/additional_features/GLDRUB_TOM_features.csv'
)
processed_df = process_currency_data(
    df=brent,
    save_path='../../data/additional_features/brent_continuous_features.csv'
)

Исходные данные: 8795 строк
После фильтрации времени: 4400 строк
После генерации признаков: 4351 строк
Количество признаков: 17
Данные сохранены в: ../../data/additional_features/CNYRUB_TOM_features.csv
Исходные данные: 8573 строк
После фильтрации времени: 4386 строк
После генерации признаков: 4337 строк
Количество признаков: 17
Данные сохранены в: ../../data/additional_features/EURRUBF_features.csv
Исходные данные: 8759 строк
После фильтрации времени: 4400 строк
После генерации признаков: 4351 строк
Количество признаков: 17
Данные сохранены в: ../../data/additional_features/GLDRUB_TOM_features.csv
Исходные данные: 13376 строк
После фильтрации времени: 4398 строк
После генерации признаков: 4349 строк
Количество признаков: 19
Данные сохранены в: ../../data/additional_features/brent_continuous_features.csv


### 2.2.1 Считвание и проверка

In [29]:
data_features_usd = pd.read_csv("../../data/additional_features/USDRUBF_features.csv")
data_features_usd.head(3)

,begin,open,close,high,low,value,volume,ticker,MA_20,MA_50,RSI_14,MACD,MACD_SIGNAL,VOLATILITY_20,CHANGE_1D,CHANGE_5D,RANGE_POSITION
0,2022-06-14 18:00:00,62.55,62.35,63.73,62.32,0,712,USDRUBF,63.0455,63.7202,43.727579,-0.507179,-0.412100,0.149285,-0.001121,0.006457,0.361290
1,2022-06-15 10:00:00,59.76,62.27,62.98,59.76,0,896,USDRUBF,62.9840,63.6632,43.129585,-0.491051,-0.427891,0.147272,-0.001283,-0.001443,0.335484
2,2022-06-15 12:00:00,62.30,62.54,62.79,62.30,0,467,USDRUBF,62.9155,63.6274,45.822476,-0.451281,-0.432569,0.146047,0.004336,0.014436,0.422581


In [27]:
data_features_usd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4349 entries, 0 to 4348
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   begin           4349 non-null   object 
 1   open            4349 non-null   float64
 2   close           4349 non-null   float64
 3   high            4349 non-null   float64
 4   low             4349 non-null   float64
 5   value           4349 non-null   int64  
 6   volume          4349 non-null   int64  
 7   ticker          4349 non-null   object 
 8   MA_20           4349 non-null   float64
 9   MA_50           4349 non-null   float64
 10  RSI_14          4349 non-null   float64
 11  MACD            4349 non-null   float64
 12  MACD_SIGNAL     4349 non-null   float64
 13  VOLATILITY_20   4349 non-null   float64
 14  CHANGE_1D       4349 non-null   float64
 15  CHANGE_5D       4349 non-null   float64
 16  RANGE_POSITION  4349 non-null   float64
dtypes: float64(13), int64(2), object(